# 기본 03 · OCR 텍스트에서 필드 추출

`03_fields.py`의 구현을 단계별로 정리합니다. `lesson.json`에 저장된 정상 문서 3개의 OCR 기록에서 발행일, 공급자, 총액을 추출하고 OCR 원문과 함께 JSON으로 저장합니다.

Python 표준 라이브러리만 사용합니다. 저장소 내부에서 위에서 아래로 실행하세요. 마지막 저장 셀은 `01_basic/p03_fields.json`을 덮어씁니다. OCR 엔진을 새로 실행하지 않습니다.


## 1. 입력 데이터 읽기

노트북이 있는 `docs` 폴더나 저장소 내부 어디에서 실행해도 실습 경로를 찾습니다. `documents`에서 `doc01`, `doc02`, `doc03`의 `ocr.text`를 사용합니다.


In [1]:
from pathlib import Path
import json
import re

start = Path.cwd().resolve()
project = next((p for p in [start, *start.parents]
                if (p / '20261109/01_basic/03_fields.py').is_file()), None)
if project is None:
    raise FileNotFoundError('저장소 내부에서 실행하세요.')
base = project / '20261109/01_basic'
docs = json.loads((base / 'lesson.json').read_text(encoding='utf-8'))['documents']
print(docs['doc01']['ocr']['text'])


INVOICE DOCO1
DATE: 2026-11-01
SUPPLIER: BLUE CAFE
ITEM QTY PRICE AMOUNT
TEA 2 3000 6000
COFFEE 3 5000 15000
CAKE 1 6000 6000
TOTAL: 27000
CURRENCY: KRW


## 2. 정규식으로 필드 추출

| 필드 | 정규식 | 처리 |
|---|---|---|
| 발행일 (B03) | `^DATE: (\d{4}-\d{1,2}-\d{1,2})$` | 날짜 문자열 추출 |
| 공급자 | `^SUPPLIER: (.+)$` | 공급자 이름 추출 |
| 총액 (B04) | `^TOTAL: (\d+)$` | 추출 후 `int()`로 정수 변환 |

`re.MULTILINE`은 `^`와 `$`가 각 줄의 시작과 끝에 대응하게 합니다. 괄호로 묶은 값은 `.group(1)`로 가져옵니다. `\d{4}`는 숫자 4자리, `\d{1,2}`는 숫자 1~2자리, `\d+`는 숫자 1개 이상을 뜻합니다.

원본과 동일하게 정상 입력을 가정합니다. 필드가 없거나 형식이 다르면 `re.search()`가 `None`을 반환하므로 `.group(1)`에서 오류가 납니다. 날짜 패턴은 자릿수만 검사하며 실제 달력상 유효성은 검사하지 않습니다.


In [2]:
records = []
for name in ['doc01', 'doc02', 'doc03']:
    text = docs[name]['ocr']['text']
    issued_at = re.search(
        r'^DATE: (\d{4}-\d{1,2}-\d{1,2})$', text, re.MULTILINE
    ).group(1)
    supplier = re.search(r'^SUPPLIER: (.+)$', text, re.MULTILINE).group(1)
    total_amount = int(re.search(r'^TOTAL: (\d+)$', text, re.MULTILINE).group(1))
    records.append({
        'document_id': name,
        'issued_at': issued_at,
        'supplier': supplier,
        'total_amount': total_amount,
        'currency': 'KRW',
        'ocr_source': 'recorded Tesseract',
        'raw_text': text,
    })
    print(name, issued_at, supplier, total_amount)


doc01 2026-11-01 BLUE CAFE 27000
doc02 2026-11-02 PINE BOOKS 18000
doc03 2026-11-03 ALPHA STUDIO 33000


## 3. 결과 구조 확인

`currency`와 `ocr_source`는 코드에서 지정한 고정값입니다. `raw_text`에는 OCR 원문을 보존해 추출 결과와 비교할 수 있게 합니다. 총액은 문자열이 아닌 정수로 저장됩니다.


In [3]:
print(json.dumps(records[0], ensure_ascii=False, indent=2))


{
  "document_id": "doc01",
  "issued_at": "2026-11-01",
  "supplier": "BLUE CAFE",
  "total_amount": 27000,
  "currency": "KRW",
  "ocr_source": "recorded Tesseract",
  "raw_text": "INVOICE DOCO1\nDATE: 2026-11-01\nSUPPLIER: BLUE CAFE\nITEM QTY PRICE AMOUNT\nTEA 2 3000 6000\nCOFFEE 3 5000 15000\nCAKE 1 6000 6000\nTOTAL: 27000\nCURRENCY: KRW"
}


## 4. JSON 저장

`ensure_ascii=False`로 한글을 그대로 기록하고, `indent=2`로 들여쓰기합니다. 출력 위치와 형식은 `03_fields.py`와 같습니다.


In [4]:
output_path = base / 'p03_fields.json'
output_path.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding='utf-8')
print('저장 완료:', output_path.name)


저장 완료: p03_fields.json
